# Stage 1 - Step 4: Image-derived prototypes (our chosen Option A)

This is our second baseline (the assignment lets us pick one of two - we went with
this one over zero-shot CLIP, see the reasoning in our notes: no new model needed,
and zero-shot CLIP's own paper shows it's weak on textures and aircraft anyway).

No training happens here at all - it's a "nearest class mean" classifier:

1. L2-normalize every feature (so raw magnitude differences between images don't
   distort the similarity comparison - two images of the same class shouldn't be
   judged "different" just because one produced a larger-magnitude feature vector)
2. For each class, average its (normalized) training examples into one prototype
   vector, then re-normalize that average
3. Classify a test image by whichever prototype it's most cosine-similar to

Because both the features and the prototypes end up unit-length, cosine similarity
is just a dot product - no need for a separate similarity formula, it falls out for
free.

Same 3 (encoder, dataset) combinations as the linear probe: ResNet-18 on DTD and
Aircraft, DINOv2 on Aircraft. Same 5-shot/10-shot subset seeds {0,1,2}. The **full**
setting is different from the linear probe though - since there's nothing stochastic
to train, the assignment only wants **one run** for full (no 3 seeds needed, there's
no randomness left once you're averaging over literally every training image).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/cvlab_stage1'
CACHE_ROOT = os.path.join(PROJECT_ROOT, 'features')
RESULTS_ROOT = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_ROOT, exist_ok=True)

print('reading cached features from:', CACHE_ROOT)
print('writing results to:', RESULTS_ROOT)

reading cached features from: /content/drive/MyDrive/cvlab_stage1/features
writing results to: /content/drive/MyDrive/cvlab_stage1/results


In [3]:
import torch
import numpy as np
import pandas as pd

def load_features(encoder, dataset, split):
    path = os.path.join(CACHE_ROOT, f'{encoder}_{dataset}_{split}.pt')
    d = torch.load(path)
    return d['features'], d['labels'], d['classes']

combos = [('resnet18', 'dtd'), ('resnet18', 'aircraft'), ('dinov2', 'aircraft')]

## The actual method - 3 small functions

This is deliberately about as simple as code gets. `l2_normalize` does the
normalizing, `compute_prototypes` builds one mean vector per class, and
`classify_by_prototype` just does a dot product against every prototype and takes
the argmax - literally the nearest-centroid rule, just in a normalized (cosine)
space instead of raw Euclidean space.

In [4]:
def l2_normalize(x, dim=1, eps=1e-8):
    return x / x.norm(dim=dim, keepdim=True).clamp(min=eps)

def compute_prototypes(features, labels, num_classes):
    """mu_c = normalize( mean_{i in class c} normalize(z_i) ) - matches the assignment's formula exactly."""
    normed = l2_normalize(features)
    prototypes = []
    for c in range(num_classes):
        mask = (labels == c)
        class_mean = normed[mask].mean(dim=0)
        prototypes.append(class_mean)
    prototypes = torch.stack(prototypes)
    return l2_normalize(prototypes)

def classify_by_prototype(features, prototypes):
    normed = l2_normalize(features)
    sims = normed @ prototypes.T   # cosine similarity, since both sides are unit-norm
    return sims.argmax(dim=1)

def make_kshot_subset(features, labels, k, seed):
    """Same balanced K-per-class sampling as the linear probe notebook, same seed convention."""
    rng = np.random.RandomState(seed)
    labels_np = labels.numpy()
    classes = np.unique(labels_np)
    selected_idx = []
    for c in classes:
        idx_c = np.where(labels_np == c)[0]
        if len(idx_c) < k:
            raise ValueError(f'class {c} only has {len(idx_c)} images, need {k}')
        chosen = rng.choice(idx_c, size=k, replace=False)
        selected_idx.extend(chosen.tolist())
    selected_idx = np.array(selected_idx)
    return features[selected_idx], labels[selected_idx]

## Running the sweep

For each combination: 5-shot x 3 seeds, 10-shot x 3 seeds, and **one** full-data run.
That's 7 runs per combination, 21 total - and each one is just a mean + a matmul, so
this whole notebook should finish in seconds, not minutes.

We also save the **full-data prototypes** for each combination to disk - notebook 5
needs these later for the required feature-visualization plot (test features shown
together with their class prototypes).

In [5]:
K_SETTINGS = [5, 10, 'full']
SEEDS = [0, 1, 2]

all_results = []
full_prototypes = {}  # saved for notebook 5's feature visualization

for encoder, dataset in combos:
    print(f'=== {encoder} / {dataset} ===')
    train_feats, train_labels, classes = load_features(encoder, dataset, 'train')
    test_feats, test_labels, _ = load_features(encoder, dataset, 'test')
    num_classes = len(classes)

    for K in K_SETTINGS:
        if K == 'full':
            # only one run - no subsampling seed to vary, the assignment is explicit
            # that the full-data prototype result only needs one run
            prototypes = compute_prototypes(train_feats, train_labels, num_classes)
            preds = classify_by_prototype(test_feats, prototypes)
            acc = (preds == test_labels).float().mean().item()
            all_results.append({
                'encoder': encoder, 'dataset': dataset, 'K': 'full', 'seed': None,
                'n_train': len(train_labels), 'test_acc': acc,
            })
            print(f'  K=full            n_train={len(train_labels):5d}  test_acc={acc:.4f}')

            full_prototypes[f'{encoder}_{dataset}'] = {
                'prototypes': prototypes, 'classes': classes,
            }
        else:
            for seed in SEEDS:
                sub_feats, sub_labels = make_kshot_subset(train_feats, train_labels, k=K, seed=seed)
                prototypes = compute_prototypes(sub_feats, sub_labels, num_classes)
                preds = classify_by_prototype(test_feats, prototypes)
                acc = (preds == test_labels).float().mean().item()
                all_results.append({
                    'encoder': encoder, 'dataset': dataset, 'K': str(K), 'seed': seed,
                    'n_train': len(sub_labels), 'test_acc': acc,
                })
                print(f'  K={str(K):5s} seed={seed}  n_train={len(sub_labels):5d}  test_acc={acc:.4f}')

print()
print(f'Done. {len(all_results)} total runs (expected 21).')
assert len(all_results) == 21

=== resnet18 / dtd ===
  K=5     seed=0  n_train=  235  test_acc=0.4697
  K=5     seed=1  n_train=  235  test_acc=0.4394
  K=5     seed=2  n_train=  235  test_acc=0.4745
  K=10    seed=0  n_train=  470  test_acc=0.5085
  K=10    seed=1  n_train=  470  test_acc=0.5319
  K=10    seed=2  n_train=  470  test_acc=0.5106
  K=full            n_train= 1880  test_acc=0.5878
=== resnet18 / aircraft ===
  K=5     seed=0  n_train=  500  test_acc=0.1599
  K=5     seed=1  n_train=  500  test_acc=0.1668
  K=5     seed=2  n_train=  500  test_acc=0.1608
  K=10    seed=0  n_train= 1000  test_acc=0.1974
  K=10    seed=1  n_train= 1000  test_acc=0.2076
  K=10    seed=2  n_train= 1000  test_acc=0.1950
  K=full            n_train= 3334  test_acc=0.2520
=== dinov2 / aircraft ===
  K=5     seed=0  n_train=  500  test_acc=0.2457
  K=5     seed=1  n_train=  500  test_acc=0.2352
  K=5     seed=2  n_train=  500  test_acc=0.2355
  K=10    seed=0  n_train= 1000  test_acc=0.2796
  K=10    seed=1  n_train= 1000  test

## Saving results and prototypes

In [6]:
results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(RESULTS_ROOT, 'prototype_results.csv'), index=False)

torch.save(full_prototypes, os.path.join(CACHE_ROOT, 'full_prototypes.pt'))

print('saved prototype_results.csv to', RESULTS_ROOT)
print('saved full_prototypes.pt to', CACHE_ROOT)
results_df

saved prototype_results.csv to /content/drive/MyDrive/cvlab_stage1/results
saved full_prototypes.pt to /content/drive/MyDrive/cvlab_stage1/features


,encoder,dataset,K,seed,n_train,test_acc
0,resnet18,dtd,5,0.0,235,0.469681
1,resnet18,dtd,5,1.0,235,0.439362
2,resnet18,dtd,5,2.0,235,0.474468
3,resnet18,dtd,10,0.0,470,0.508511
4,resnet18,dtd,10,1.0,470,0.531915
5,resnet18,dtd,10,2.0,470,0.510638
6,resnet18,dtd,full,NaN,1880,0.587766
7,resnet18,aircraft,5,0.0,500,0.159916
8,resnet18,aircraft,5,1.0,500,0.166817
9,resnet18,aircraft,5,2.0,500,0.160816


## Quick comparison against the linear probe

Worth a quick look now: how does this simple, training-free method compare to the
27-run linear probe sweep from notebook 3? This isn't one of the required deliverable
plots (that's notebook 5's job to make properly), just a sanity gut-check now.

In [7]:
lp_path = os.path.join(RESULTS_ROOT, 'linear_probe_results.csv')
if os.path.exists(lp_path):
    lp_df = pd.read_csv(lp_path)
    lp_summary = lp_df.groupby(['encoder', 'dataset', 'K'])['test_acc'].mean().reset_index()
    lp_summary['method'] = 'linear_probe'

    proto_summary = results_df.groupby(['encoder', 'dataset', 'K'])['test_acc'].mean().reset_index()
    proto_summary['method'] = 'prototype'

    combined = pd.concat([lp_summary, proto_summary], ignore_index=True)
    pivot = combined.pivot_table(index=['encoder', 'dataset', 'K'], columns='method', values='test_acc')
    print(pivot)
else:
    print('linear_probe_results.csv not found yet - run notebook 3 first if you want this comparison')
    print()
    print(results_df.groupby(['encoder', 'dataset', 'K'])['test_acc'].mean())

method                  linear_probe  prototype
encoder  dataset  K                            
dinov2   aircraft 10        0.519052   0.276028
                  5         0.383138   0.238824
                  full      0.670567   0.342334
resnet18 aircraft 10        0.275528   0.200020
                  5         0.201520   0.162516
                  full      0.369937   0.252025
         dtd      10        0.523227   0.517021
                  5         0.447695   0.461170
                  full      0.628369   0.587766


## What to expect / think about here

Generally, expect the linear probe to edge out the prototype method, especially as K
grows - it's actually learning a decision boundary rather than just averaging, so it
can exploit more of what's discriminative in the feature space. The prototype method
tends to close the gap (or even compete) in the very-low-shot regime, since there's
so little data that a trained classifier doesn't have much advantage over "just
average what you've got."

If you see the *opposite* pattern (prototypes clearly beating the linear probe
across the board), that's worth double-checking rather than assuming it's a real
result - it could mean the linear probe is underfitting (e.g. not enough epochs, or
the fixed lr isn't suited to a particular K), which per the assignment is fine to
notice and adjust ("if the suggested setting behaves poorly, adjust it... and report
the changes").

## Before moving to notebook 5

Check that:
- [ ] `prototype_results.csv` has 21 rows in Drive `results/`
- [ ] `full_prototypes.pt` exists in Drive `features/`
- [ ] Accuracy generally increases with more shots, same as the linear probe showed

Next: **notebook 5** pulls together everything from notebooks 3 and 4 into the
actual required deliverables - accuracy table, accuracy-vs-K plot with error bars,
confusion matrices, and the PCA/t-SNE feature visualization.